# [Chapter 9 — Practical Issues in Fitting, §9.1] Pitfall 1: Accounting for Uncertainty in the Assumed Susceptible Population

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 9 — Practical Issues in Fitting
**Considerations developed:** 8, 9
**Estimated runtime:** ≤ 30s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Demonstrates how a wrong $N^*$ assumption produces a biased $\hat\lambda$-based $\mathcal{R}_0$. Quantifies the bias as a function of $N^*$ error and shows how the $\hat\alpha$ estimator avoids this pitfall.

## What you should already know
Chapter 8 central comparison.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance
set_seed_chapter_08()
book_style()


## The pitfall

Many fitting workflows assume $S_0 = N$ (entire population susceptible) without checking. This produces a biased $\hat\lambda$ — and a correspondingly biased $\mathcal{R}_0$ — when prior immunity is present.

We sweep the true initial susceptible fraction $F_0$ from 0.5 to 1.0, holding all other parameters fixed, and observe how naively applying $\hat\lambda = J/N$ misleads.

In [ ]:
params = baseline_chapter_08()
true_R0 = params['c_I'] * params['beta'] * params['tau_R']

F0_range = np.linspace(0.5, 1.0, 11)
R0_lambda_naive = []
R0_alpha_correct = []

for F0 in F0_range:
    p = params.copy()
    p['S0'] = F0
    p['I0'] = 0.001
    p['R0_init'] = 1.0 - F0 - 0.001
    res = integrate_sir_i(p)
    t, S, I = res['t'], res['S'], res['I']
    J = p['c_I'] * p['beta'] * (S / p['N']) * I

    mask = (t > 5) & (t < 20)
    # Naive lambda: assume S* = N
    lambda_naive = J[mask].mean() / p['N']
    R0_lambda_naive.append(lambda_naive * p['tau_R'] / (I[mask].mean() / p['N']))
    # alpha: uses I directly, then converts using true S
    alpha_hat = J[mask].mean() / I[mask].mean()
    R0_alpha_correct.append(alpha_hat * p['tau_R'] / (S[mask].mean() / p['N']))

R0_lambda_naive = np.array(R0_lambda_naive)
R0_alpha_correct = np.array(R0_alpha_correct)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(F0_range, R0_lambda_naive, 'o-', color=BOOK_COLORS['lambda_hat'], label=r'$\hat{R}_0$ from naive $\hat\lambda$ (assumes $S^*=N$)')
ax.plot(F0_range, R0_alpha_correct, 's-', color=BOOK_COLORS['alpha_hat'], label=r'$\hat{R}_0$ from $\hat\alpha$ (uses observed $I$, $S$)')
ax.axhline(true_R0, ls='--', color=BOOK_COLORS['neutral'], lw=1, alpha=0.7, label=f'True R_0 = {true_R0:.1f}')
ax.set_xlabel('True initial susceptible fraction $F_0 = S_0/N$')
ax.set_ylabel(r'$\hat{R}_0$ estimate')
ax.set_title('Pitfall 1: bias from wrong $S^*$ assumption')
ax.legend()
plt.tight_layout()
plt.show()

bias_at_F0_07 = abs(R0_lambda_naive[F0_range.argmin()*0 + (F0_range == 0.7).argmax()] - true_R0) / true_R0
print(f"\nWith F_0 = 0.7 (30% prior immunity), naive lambda gives bias of ~{bias_at_F0_07*100:.0f}%")
print("alpha-based estimate is unbiased (uses observed S directly).")
